# Zero-shot transfer

> Score the OSSL models on every ring-trial instrument, without adapting them

In [1]:
#| default_exp zero_shot

In [2]:
#| export
import numpy as np, pandas as pd
from soilspecdl.all import predict, regression_scores
from ossl_rt_transfer.pretrain import rt_wns, log_props, snv_d1

/Users/franckalbinet/pro/dev/soilspec/ossl-rt-transfer/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [3]:
from fastcore.test import *
from soilspecdata.datasets.ringtrial import load_ringtrial
from ossl_rt_transfer.data import rt_path, load_instruments, ossl_props
from ossl_rt_transfer.pretrain import load_pretrained

The cells that read the ring-trial data are marked `#| rtdata`. The outputs on this page come from a local run.

## Scoring

In [4]:
#| export
def zero_shot(
    model, # Pretrained model, such as from `load_pretrained`
    spectra, # `Spectra` of one ring-trial instrument, on `rt_wns`
    prop, # Ring-trial property, a key of `ossl_props`
)->dict: # Number of soils, and `regression_scores`
    "Score `model` on the soils of one instrument that have a value of `prop`, with no adaptation to the instrument"
    X, y, _ = spectra.xy(prop)
    y = np.log1p(y.ravel()) if prop in log_props else y.ravel()
    p = predict(model, snv_d1().fit_transform(X), rt_wns)
    return {'n': len(y), **regression_scores(y, p, inv=np.expm1 if prop in log_props else None)}

`zero_shot` preprocesses the spectra as in pretraining, and scores the predictions against the measured values. For K, R², `r` and CCC are on `log1p(K)`, as in pretraining, and RMSE and bias are in cmolc/kg. The 38 ring-trial soils that are in OSSL were left out of pretraining, so every soil is a test soil.

In [5]:
#| rtdata
rtd = load_ringtrial(rt_path())
models = {p: load_pretrained(f'ossl_{p}_snv_d1') for p in ossl_props}
zs = pd.DataFrame([{'instrument': k, 'property': p, **zero_shot(models[p], s, p)} for k, s in rtd.items() for p in ossl_props])
test_eq(len(zs), 21 * 3)
clusters = load_instruments().cluster.reindex(list(rtd)).astype('Int64')
by_inst = lambda col: zs.pivot(index='instrument', columns='property', values=col).assign(cluster=clusters).sort_values(['cluster', 'pH_H20'])

## Results

R² by instrument, grouped by the spectral clusters of Safanelli et al. (2023). Hokudai is not in the instrument table, so it has no cluster:

In [6]:
#| rtdata
by_inst('R²').round(2)

property,clay_perc,pH_H20,potassium_cmolkg,cluster
instrument,,,,
UWisc_fine,0.32,0.77,0.22,1
Lesotho,-0.12,0.16,-0.26,2
IAEA_aug2022,0.37,0.59,-0.89,2
UIUC,0.29,0.61,-1.59,2
Argonne,0.69,0.72,0.50,2
OSU,0.61,0.74,-0.23,2
UGhent,0.33,0.76,-0.30,2
Woodwell_vertex,0.53,0.63,-0.26,3
USP,0.41,0.66,-0.57,3


Correlation `r`:

In [7]:
#| rtdata
by_inst('r').round(2)

property,clay_perc,pH_H20,potassium_cmolkg,cluster
instrument,,,,
UWisc_fine,0.80,0.88,0.71,1
Lesotho,0.28,0.77,0.43,2
UIUC,0.68,0.86,0.66,2
OSU,0.81,0.88,0.75,2
UGhent,0.71,0.89,0.53,2
Argonne,0.83,0.89,0.77,2
IAEA_aug2022,0.70,0.90,0.69,2
USP,0.69,0.85,0.69,3
Woodwell_vertex,0.75,0.86,0.70,3


Bias, as mean prediction minus mean measured value: clay in percentage points, pH in pH units, K in cmolc/kg:

In [8]:
#| rtdata
by_inst('bias').round(2)

property,clay_perc,pH_H20,potassium_cmolkg,cluster
instrument,,,,
UWisc_fine,6.27,0.08,0.17,1
Lesotho,-3.61,-0.80,-0.16,2
IAEA_aug2022,-4.10,-0.55,0.46,2
UIUC,4.21,-0.37,0.59,2
Argonne,-0.04,-0.24,0.05,2
OSU,1.93,-0.06,0.34,2
UGhent,3.28,0.10,0.23,2
Agrocares,-0.47,-0.25,0.19,3
Woodwell_vertex,1.90,-0.24,0.30,3


The median over the 21 instruments, and the KSSL instrument, whose spectra are like those of OSSL:

In [9]:
#| rtdata
cols = ['R²', 'r', 'RMSE', 'bias']
pd.concat({'median': zs.groupby('property')[cols].median(), 'KSSL': zs[zs.instrument == 'KSSL'].set_index('property')[cols]}, axis=1).round(2)

median                    KSSL                  
                     R²     r  RMSE  bias    R²     r  RMSE  bias
property                                                         
clay_perc          0.53  0.81  8.06  1.93  0.65  0.91  6.89  4.51
pH_H20             0.72  0.88  0.65 -0.13  0.82  0.92  0.52 -0.13
potassium_cmolkg  -0.17  0.74  0.50  0.23  0.15  0.78  0.52  0.15